# Module 1: Building a Multi-Agent Customer Service System

## Overview

In this module, you'll build a production-ready multi-agent system for e-commerce customer service using the Strands Agent SDK with **MCP (Model Context Protocol)** tools. 

### MCP Architecture
The Order and Product agents use **MCP (Model Context Protocol)** to connect to separate MCP servers that expose tools. This provides:
- **Decoupling**: Tools run as separate processes
- **Reusability**: MCP servers can be shared across applications
- **Standardization**: Follows the MCP protocol for tool discovery

### Learning Objectives
1. Build specialized agents with Strands SDK and MCP tools
2. Create an orchestrator for intelligent request routing
3. Understand cost optimization with mixed LLM strategy
4. Test the multi-agent system with various queries


## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [9]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3 mcp

In [10]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['ORDERS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-orders-table')['Parameter']['Value']
    os.environ['ACCOUNTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-accounts-table')['Parameter']['Value']
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"✓ Orders Table: {os.environ['ORDERS_TABLE_NAME']}")
    print(f"✓ Accounts Table: {os.environ['ACCOUNTS_TABLE_NAME']}")
    print(f"✓ Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['ORDERS_TABLE_NAME'] = 'ecommerce-workshop-orders'
    os.environ['ACCOUNTS_TABLE_NAME'] = 'ecommerce-workshop-accounts'
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

AWS Region: us-east-1
✓ Orders Table: ecommerce-workshop-orders
✓ Accounts Table: ecommerce-workshop-accounts
✓ Products Table: ecommerce-workshop-products


## Step 2: Understanding MCP Tools

The Order and Product agents use **MCP (Model Context Protocol)** to access tools. Instead of importing Python functions directly, the agents connect to MCP servers that expose tools.

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)
```

### MCP Servers in this Module

| Server | File | Tools |
|--------|------|-------|
| Order Service | `mcp_servers/order_mcp_server.py` | get_order_status, track_shipment, process_return, modify_order, get_customer_orders |
| Product Service | `mcp_servers/product_mcp_server.py` | search_products, get_product_details, check_inventory, get_product_recommendations, compare_products, get_return_policy |

### How Agents Connect to MCP

```python
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Configure MCP server connection
server_params = StdioServerParameters(
    command="python",
    args=["mcp_servers/order_mcp_server.py"]
)

# Connect and get tools
mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()
tools = mcp_client.list_tools_sync()

# Create agent with MCP tools
agent = Agent(model=model, tools=tools)
```

In [3]:
# Verify MCP servers exist
# Add agents to path
sys.path.insert(0, 'agents')
import os
from pathlib import Path

mcp_servers_dir = Path('mcp_servers')
order_server = mcp_servers_dir / 'order_mcp_server.py'
product_server = mcp_servers_dir / 'product_mcp_server.py'

print("MCP Servers:")
print(f"  Order Server: {order_server} - {'✓ exists' if order_server.exists() else '✗ missing'}")
print(f"  Product Server: {product_server} - {'✓ exists' if product_server.exists() else '✗ missing'}")

MCP Servers:
  Order Server: mcp_servers/order_mcp_server.py - ✓ exists
  Product Server: mcp_servers/product_mcp_server.py - ✓ exists


## Step 3: Build Specialized Agents

Now let's create our specialized agents. The Order and Product agents use **MCP tools** from separate MCP servers, while the Account agent uses native tools.

The MCP servers are located in `mcp_servers/` directory:
- `order_mcp_server.py` - Exposes order management tools
- `product_mcp_server.py` - Exposes product catalog tools

In [12]:
from strands import Agent
from strands.models import BedrockModel

# Model configuration using global cross-region inference profiles
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Pricing per 1M tokens (approximate, check AWS pricing page for latest)
# https://aws.amazon.com/bedrock/pricing/
HAIKU_INPUT_PRICE = 0.80    # $0.80 per 1M input tokens
HAIKU_OUTPUT_PRICE = 4.00   # $4.00 per 1M output tokens
SONNET_INPUT_PRICE = 3.00   # $3.00 per 1M input tokens  
SONNET_OUTPUT_PRICE = 15.00 # $15.00 per 1M output tokens

print(f"Haiku Model: {HAIKU_MODEL_ID}")
print(f"Sonnet Model: {SONNET_MODEL_ID}")
print("\nOrder and Product agents use MCP tools via MCPClient")

Haiku Model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Sonnet Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0

Order and Product agents use MCP tools via MCPClient


### 3.1 Create Order Agent

In [5]:
from order_agent import create_order_agent

order_agent = create_order_agent(region=REGION)
print(f"Order Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from order_mcp_server.py")

Order Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Tools loaded via MCP from order_mcp_server.py


In [6]:
# Test Order Agent directly
response = order_agent("What's the status of order ORD-2024-10002?")
print("Order Agent Response:")
print(response)


Tool #1: get_order_status
Your order **ORD-2024-10002** is currently **shipped** and on its way to you!

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number provided. Would you like me to get more detailed tracking information or help you with anything else?Order Agent Response:
Great! Here's the status of your order:

**Order ID:** ORD-2024-10002  
**Status:** Shipped ✓

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Deliver

### 3.2 Create Product Agent

In [7]:
from product_agent import create_product_agent

product_agent = create_product_agent(region=REGION)
print(f"Product Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from product_mcp_server.py")

Product Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Tools loaded via MCP from product_mcp_server.py


In [8]:
# Test Product Agent directly
response = product_agent("Do you have any wireless headphones under $100?")
print("Product Agent Response:")
print(response)

I'll search for wireless headphones under $100 for you.
Tool #1: search_products
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones - $79.99** (PROD-001)
- **Type:** Over-ear wireless headphones
- **Key Features:**
  - Active noise cancellation
  - 40-hour battery life
  - Comfortable memory foam ear cushions
- **Stock Status:** In stock ✓

This is an excellent value at under $80! The 40-hour battery life is particularly impressive for the price point.

I also found another option that's slightly above your budget:
- **Noise Canceling Earbuds - $149.99** (PROD-055) - True wireless earbuds with adaptive noise cancellation and 8-hour battery per charge

Would you like more details about the $79.99 headphones, or would you like me to check inventory availability or compare these options?Product Agent Response:
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones - $79.99** (PROD-001)
- **Type:** Over-ear wireless 

### 3.3 Create Account Agent

In [9]:
from account_agent import create_account_agent

account_agent = create_account_agent(region=REGION)
print(f"Account Agent created with model: {HAIKU_MODEL_ID}")

Account Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [10]:
# Test Account Agent directly
response = account_agent("What are the benefits of Gold membership?")
print("Account Agent Response:")
print(response)


Tool #1: get_membership_benefits
Great question! Here are the benefits of **Gold membership**:

✨ **Gold Membership Benefits:**

- **Free Shipping on Orders Over $25** - Lower threshold than Standard tier
- **45-Day Return Window** - Extended time to return items
- **Early Access to Sales** - Get first dibs on exclusive sales and promotions
- **Earn 1.5x Points** - Earn 1.5 reward points for every $1 spent (vs. standard rate)
- **Birthday Discount** - Receive 15% off during your birthday month

**How Gold Compares:**
- **Standard Tier:** Free shipping over $50, standard return window, no early sale access
- **Gold Tier:** Free shipping over $25, 45-day returns, early sale access, bonus points
- **Platinum Tier:** Free shipping on all orders, priority support, premium benefits

Gold membership is a great middle-ground option if you're looking for better shipping benefits and want to earn rewards faster! If you're interested in upgrading or have questions about whether Gold is right for

## Step 4: Create the Orchestrator

The Orchestrator uses Claude Sonnet 4.5 (larger model) for complex reasoning and routes requests to specialized agents.

In [11]:
from orchestrator import MultiAgentCustomerService

# Create the multi-agent system
customer_service = MultiAgentCustomerService(region=REGION)
print("Multi-Agent Customer Service System initialized!")
print(f"- Orchestrator: {SONNET_MODEL_ID}")
print(f"- Order Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Product Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Account Agent: {HAIKU_MODEL_ID} (native tools)")

Multi-Agent Customer Service System initialized!
- Orchestrator: global.anthropic.claude-sonnet-4-5-20250929-v1:0
- Order Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (MCP tools)
- Product Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (MCP tools)
- Account Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (native tools)


## Step 5: Test the Multi-Agent System

Let's test various customer scenarios to see how the orchestrator routes requests.

In [12]:
# Test Case 1: Order inquiry (should route to Order Agent)
print("="*60)
print("Test Case 1: Order Inquiry")
print("="*60)

response = customer_service.chat("Where is my order ORD-2024-10002?")
print(f"Customer: Where is my order ORD-2024-10002?")
print(f"\nAgent: {response}")

Test Case 1: Order Inquiry

Tool #1: delegate_to_order_agent

Tool #1: get_order_status

Tool #2: track_shipment
Great news! Your order **ORD-2024-10002** has been **shipped** and is on its way to you.

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Tracking Link:** [Track your package on UPS website](https://www.ups.com/track?tracknum=1Z999AA10123456784)

**Items in Your Order:**
- 1x Smart Watch Pro ($299.99)
- 2x Watch Band - Leather ($29.99 each)

**Total:** $359.97

You can track your package in real-time using your UPS tracking number here: https://www.ups.com/track?tracknum=1Z999AA10123456784

Is there anything else you'd like to know about your order?Great news! Your order **ORD-2024-10002** is on its way! 📦

**Order Status:** Shipped

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Tracking Link:** [Track your packag

In [13]:
# Test Case 2: Product search (should route to Product Agent)
print("="*60)
print("Test Case 2: Product Search")
print("="*60)

response = customer_service.chat("I'm looking for a good gaming keyboard with RGB lighting")
print(f"Customer: I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nAgent: {response}")

Test Case 2: Product Search

Tool #2: delegate_to_product_agent
I'll search for gaming keyboards with RGB lighting for you.
Tool #1: search_products
Great! I found a gaming keyboard with RGB lighting. Let me get more detailed information about it.
Tool #2: get_product_details
Perfect! Here's what I found:
Perfect! Here's what I found:

## **Gaming Mechanical Keyboard RGB** (PROD-077)
**Price:** $159.99 | **In Stock** ✓

### Key Features:
- **Lighting:** Per-key RGB with 16.8M colors - fully customizable
- **Switches:** Hot-swappable Cherry MX compatible switches (easy to replace)
- **Connection:** USB-C detachable cable for portability
- **Anti-Ghosting:** Full N-key rollover for responsive gaming
- **Extras:** Dedicated media keys + volume wheel, detachable wrist rest
- **Layout:** Full-size keyboard

### Additional Info:
- **Rating:** 4.6 out of 5 stars
- **Warranty:** 2 years
- **Return Policy:** 30-day return policy
- **Stock:** 60 units available

This is a solid choice for gaming

In [14]:
# Test Case 3: Account management (should route to Account Agent)
print("="*60)
print("Test Case 3: Account Management")
print("="*60)

response = customer_service.chat("I need to reset my password for john.smith@email.com")
print(f"Customer: I need to reset my password for john.smith@email.com")
print(f"\nAgent: {response}")

Test Case 3: Account Management

Tool #3: delegate_to_account_agent
I'll initiate a password reset for your account right away.
Tool #1: initiate_password_reset
Perfect! I've initiated the password reset process for john.smith@email.com.

✓ **Password reset link sent** to john.smith@email.com
- Check your email inbox (and spam/junk folder just in case)
- The reset link will **expire in 1 hour**, so act quickly
- Click the link to create a new password

**Important security notes:**
- Never share your reset link with anyone
- Make sure to create a strong password with a mix of letters, numbers, and special characters
- If you don't receive the email within a few minutes, check your spam folder

If you don't receive the reset email or have any other issues, please let me know and I can help further!Perfect! I've initiated the password reset process for john.smith@email.com. Here's what you need to know:

✓ **Password reset link sent** to john.smith@email.com
- Check your email inbox (and

In [15]:
# Test Case 4: Complex multi-domain query (should use multiple agents)
print("="*60)
print("Test Case 4: Complex Multi-Domain Query")
print("="*60)

response = customer_service.chat(
    "I want to return order ORD-2024-10001 because the headphones don't fit well. "
    "Can you recommend a different pair that might be more comfortable?"
)
print(f"Customer: I want to return order ORD-2024-10001 because the headphones don't fit well. Can you recommend a different pair that might be more comfortable?")
print(f"\nAgent: {response}")

Test Case 4: Complex Multi-Domain Query
I'll help you with both the return and finding more comfortable headphones. Let me start by processing your return request and then get you some recommendations.
Tool #4: delegate_to_order_agent

Tool #5: delegate_to_product_agent
II'll get some personalized recommendations for comfortable headphones based on your previous fit issues.'ll process a return request for your order ORD-2024-10001.
Tool #3: get_product_recommendations

Tool #3: process_return
Great! I found some comfortable headphone options. Let me get detailed information on the best fits for comfort:
Tool #4: get_product_details

Tool #5: get_product_details
I'm sorry, but I'm unable to process a return for order **ORD-2024-10001** at this time. 

**Reason:** The return window has expired. This order was placed 408 days ago, and we allow returns within 30 days of purchase.

**What you can do:**
Since this is outside our standard return window, I'd recommend contacting our customer s

In [16]:
# Test Case 5: Inventory check
print("="*60)
print("Test Case 5: Inventory Check")
print("="*60)

response = customer_service.chat("Is the 4K webcam PROD-088 in stock?")
print(f"Customer: Is the 4K webcam PROD-088 in stock?")
print(f"\nAgent: {response}")

Test Case 5: Inventory Check

Tool #6: delegate_to_product_agent

Tool #6: check_inventory
Unfortunately, the **4K webcam (PROD-088) is currently out of stock**. 

**Restock Information:**
- **Expected restock date:** January 15, 2025
- **Current stock:** 0 units available

**Would you like me to:**
1. Get details about the PROD-088 so you know what to expect when it's back in stock?
2. Search for alternative 4K webcams that are currently available?
3. Help you with other products in the meantime?

Let me know how I can help!Unfortunately, the **4K webcam (PROD-088) is currently out of stock**. 

**Restock Information:**
- **Expected restock date:** January 15, 2025
- **Current stock:** 0 units available

**Would you like me to:**
1. Get details about the PROD-088 so you know what to expect when it's back in stock?
2. Search for alternative 4K webcams that are currently available?
3. Help you with other products in the meantime?

Let me know how I can assist you!Customer: Is the 4K web

## Summary

In this module, you built a multi-agent customer service system using **MCP tools** that:

1. **Uses MCP tools** - Order and Product agents connect to MCP servers for tool access
2. **Optimizes costs** by using smaller models for specialized tasks
3. **Routes intelligently** using a larger model for orchestration only
4. **Handles complex queries** that span multiple domains

### MCP Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Orchestrator (Sonnet)                     │
│                   - Intent classification                    │
│                   - Request routing                          │
└─────────────────────┬───────────────┬───────────────────────┘
                      │               │
        ┌─────────────▼───┐   ┌───────▼─────────────┐
        │  Order Agent    │   │   Product Agent     │
        │   (Haiku)       │   │     (Haiku)         │
        │   MCP Tools     │   │     MCP Tools       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │ order_mcp_server│   │ product_mcp_server  │
        │    (FastMCP)    │   │     (FastMCP)       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │    DynamoDB     │   │     DynamoDB        │
        │  Orders Table   │   │   Products Table    │
        └─────────────────┘   └─────────────────────┘
```

### Key Files

| File | Description |
|------|-------------|
| `mcp_servers/order_mcp_server.py` | MCP server for order tools |
| `mcp_servers/product_mcp_server.py` | MCP server for product tools |
| `agents/order_agent.py` | Order Agent with MCP client |
| `agents/product_agent.py` | Product Agent with MCP client |
| `agents/orchestrator.py` | Multi-agent orchestrator |


### Next Steps

In **Module 2**, we'll evaluate this system with a comprehensive test dataset and establish baseline metrics for production monitoring.

In [ ]:
# Clean up MCP connections before ending
order_agent.cleanup()
product_agent.cleanup()
customer_service.cleanup()
print("MCP connections cleaned up")

# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")